# `Reporting Excel automatisé en Python`
*Oscar JOSEPH--GENESLAY*
**Data :** SuperStore Dataset
Source : *[Kaggle - SuperStore Dataset par vivek468](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final?resource=download)*

# Importation des données

In [150]:
import pandas as pd
dataset = pd.read_csv(
    "https://minio.lab.sspcloud.fr/oscar04/Superstore/Superstore.csv",
    encoding="windows-1252"
)

dataset.head(3)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


In [151]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

# Reformatage des colonnes

In [152]:
from datetime import datetime

# Retypage
dataset["Row ID"] = str(dataset["Row ID"])
dataset["Postal Code"] = str(dataset["Postal Code"])


# Transformation en format date
dataset["Order Date"] = pd.to_datetime(dataset["Order Date"], format='%d/%m/%Y', errors='coerce')
dataset["Ship Date"] = pd.to_datetime(dataset["Ship Date"], format='%d/%m/%Y', errors='coerce')

dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   object        
 1   Order ID       9994 non-null   object        
 2   Order Date     4042 non-null   datetime64[ns]
 3   Ship Date      3898 non-null   datetime64[ns]
 4   Ship Mode      9994 non-null   object        
 5   Customer ID    9994 non-null   object        
 6   Customer Name  9994 non-null   object        
 7   Segment        9994 non-null   object        
 8   Country        9994 non-null   object        
 9   City           9994 non-null   object        
 10  State          9994 non-null   object        
 11  Postal Code    9994 non-null   object        
 12  Region         9994 non-null   object        
 13  Product ID     9994 non-null   object        
 14  Category       9994 non-null   object        
 15  Sub-Category   9994 n

# Fichier du reporting

In [153]:
from openpyxl import *

## Création du fichier Excel

In [154]:
with pd.ExcelWriter("../output/reporting.xlsx", engine="openpyxl") as writer:
    dataset.to_excel(writer, sheet_name="DATA", index=False)

print("✅ reporting.xlsx créé")

✅ reporting.xlsx créé


## Chargement du reporting en mémoire

In [155]:
# Charger le classeur (workbook) en mémoire
wb = openpyxl.load_workbook('../output/reporting.xlsx')

# Création d'une feuille visualisations
ws_visualisations = wb.create_sheet(title="VISUALISATIONS")

# Définir la feuille DATA en variable python
ws_data = wb["DATA"]

# wb est un objet Python — on ne peut pas afficher son contenu avec un simple print
print(type(wb))
print('Feuilles disponibles :', wb.sheetnames)

<class 'openpyxl.workbook.workbook.Workbook'>
Feuilles disponibles : ['DATA', 'VISUALISATIONS']


# Création des KPIs

### KPIs numériques

In [156]:
# Création fonction automatisation
def numerical_kpis(cell, colonne, cell_format):
    ws_visualisations[cell] = f"=SUM(DATA!{colonne}:{colonne})"
    ws_visualisations[cell].number_format = cell_format

In [157]:
# CA
numerical_kpis("B6", "R", "#,##0.00 €")

# Profit
numerical_kpis("G6", "U", "#,##0.00 €")

# Quantité
numerical_kpis("L6", "S", "#,##0")

### TOP10 des sous-catégories avec le plus de profits

In [158]:
dataset_profits = dataset.groupby('Sub-Category')['Profit'].sum().round(1).reset_index()
dataset_profits_10 = dataset_profits.sort_values(by='Profit', ascending=False).head(10)
dataset_profits_10.head(5)

,Sub-Category,Profit
6,Copiers,55617.8
13,Phones,44515.7
0,Accessories,41936.6
12,Paper,34053.6
3,Binders,30221.8


In [159]:
from openpyxl.chart import BarChart, Reference
from openpyxl.utils.dataframe import dataframe_to_rows

In [160]:
for r_idx, row in enumerate(dataframe_to_rows(dataset_profits_10, index=False, header=True)):
    for c_idx, value in enumerate(row):
        cell = ws_data.cell(row=3 + r_idx, column=26 + c_idx, value=value)
        
        # Format Euro sur la colonne des profits (AA) sauf l'en-tête
        if c_idx == 1 and r_idx > 0:
            cell.number_format = '#,##0 €'

In [161]:
# Création du graphique à barres
chart_top10 = BarChart()
chart_top10.type = "col"
chart_top10.title = "TOP 10 du profit en fonction des sous-catégories"

# Récupération des données de profits
data_ref = Reference(ws_data, min_col=27, min_row=2, max_row=13)

# Récupération des étiquettes de sous catégories
cats_ref = Reference(ws_data, min_col=26, min_row=3, max_row=13)

# On applique au graphique
chart_top10.add_data(data_ref, titles_from_data=True)
chart_top10.set_categories(cats_ref)

# 3. PLACER LE GRAPHIQUE EN B15 DANS L'ONGLET VISUALISATIONS
ws_visualisations.add_chart(chart_top10, "B15")

In [162]:
print(dataset_profits_10.head(2))
print("Colonnes disponibles :", dataset_profits_10.columns.tolist())

   Sub-Category   Profit
6       Copiers  55617.8
13       Phones  44515.7
Colonnes disponibles : ['Sub-Category', 'Profit']


# Création de l'aspect design

### Le titre

In [163]:
ws_visualisations["L1"] = "Performances économique"
ws_visualisations["L1"].font = Font(bold=True, size=28, color="185EB8")

### KPIs numériques

In [164]:
def design_numerical_kpis(cell, nom):
    ws_visualisations[cell] = nom
    ws_visualisations[cell].font = Font(bold=True, size=16)

In [165]:
# CA
design_numerical_kpis("B5", "Chiffre d'affaires")

# Profits
design_numerical_kpis("G5", "Profits")

# Quantités
design_numerical_kpis("L5", "Volume des ventes")

# Export du reporting

In [166]:
wb.save("../output/reporting.xlsx")